# BFND step 5: does the hybrid retriever find passages nobody has labelled?

Run from a fresh runtime with **Runtime → Run all**.

**Before you start, put these in your Google Drive:**
- `BFND_step5_code.zip` in `MyDrive` (the current code)
- `MyDrive/BFND-data/data/annotations/evidence_eval_sheet.csv` (the labelled sheet: dev rows labelled, eval rows blank)
- `MyDrive/BFND-data/features/meta.csv`
- `MyDrive/BFND-data/data/evidence/evidence.csv`

**What it does:**
1. Mounts Drive and checks the files.
2. Installs the two packages the retriever needs.
3. Unzips the code and runs the tests for the evidence code.
4. Checks the sheet: dev rows must be labelled, eval rows must be blank. The run stops if not.
5. Runs `scripts/14_dev_retriever_overlap.py`: for each dev post, it compares the BM25 top-3 with the hybrid (BM25 + dense) top-3.
6. Shows the result, and lists the passages nobody has labelled yet.
7. Copies the results to `MyDrive/BFND-data/exports/step5_<time>/` (copy only).

**What it does not do:** it does not read the eval labels, does not change the sheet, and does not decide the retriever. The retriever is chosen in a later step, on dev only (D-11).

## Cell 1: mount Drive and check the files

In [1]:
import os
from google.colab import drive

drive.mount("/content/drive")
DRIVE = "/content/drive/MyDrive"
BFND = f"{DRIVE}/BFND-data"
CODE_ZIP = f"{DRIVE}/BFND_step5_code.zip"
SHEET = f"{BFND}/data/annotations/evidence_eval_sheet.csv"
META = f"{BFND}/features/meta.csv"
EVIDENCE = f"{BFND}/data/evidence/evidence.csv"

for path, what in [(CODE_ZIP, "BFND_step5_code.zip"), (SHEET, "evidence_eval_sheet.csv"),
                   (META, "features/meta.csv"), (EVIDENCE, "data/evidence/evidence.csv")]:
    assert os.path.exists(path), f"Missing {what}. Expected at: {path}"
print("Drive mounted. All four input files found.")

Mounted at /content/drive
Drive mounted. All four input files found.


## Cell 2: install the packages the retriever needs

In [2]:
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "sentence-transformers>=3.0",
                "rank-bm25>=0.2.2", "pytest"], check=True)
import sentence_transformers, rank_bm25
print("sentence-transformers", sentence_transformers.__version__, "| rank-bm25 ready")

sentence-transformers 5.7.0 | rank-bm25 ready


## Cell 3: unzip the code and run the evidence tests

In [3]:
import os, zipfile, subprocess, sys

EXTRACT = "/content/bfnd_step5"
os.makedirs(EXTRACT, exist_ok=True)
with zipfile.ZipFile(CODE_ZIP) as z:
    z.extractall(EXTRACT)

PROJECT = None
for root, dirs, files in os.walk(EXTRACT):
    if "src" in dirs and "scripts" in dirs and "configs" in dirs:
        PROJECT = root
        break
assert PROJECT is not None, "src/, scripts/ and configs/ not found in the zip"
assert os.path.exists(os.path.join(PROJECT, "scripts/14_dev_retriever_overlap.py")), \
    "scripts/14 is missing from the zip. Rebuild the zip."
print("Project folder:", PROJECT)

r = subprocess.run([sys.executable, "-m", "pytest", "-q",
                    "tests/test_evidence_eval.py", "tests/test_evidence_app.py", "tests/test_ocr_metrics.py"],
                   cwd=PROJECT, capture_output=True, text=True)
print(r.stdout[-1500:])
r.check_returncode()
print("Tests passed")

Project folder: /content/bfnd_step5/BFND_code
...................................                                      [100%]
35 passed in 0.83s

Tests passed


## Cell 4: check the sheet before the retriever runs

The run stops if a dev row is unlabelled or if any eval row has a label. The eval rows must stay blank until the retriever is frozen (D-11).

In [4]:
import pandas as pd

sheet = pd.read_csv(SHEET, encoding="utf-8-sig", dtype=str, keep_default_na=False)
print("columns:", list(sheet.columns))
print(sheet["role"].value_counts().to_string())

dev = sheet[sheet["role"] == "dev"]
eval_rows = sheet[sheet["role"] == "eval"]
dev_labelled = (dev["relevant_0_or_1"].str.strip() != "").sum()
eval_labelled = (eval_rows["relevant_0_or_1"].str.strip() != "").sum()
print(f"dev rows labelled: {dev_labelled} of {len(dev)}")
print(f"eval rows labelled: {eval_labelled} (must be 0)")

assert dev_labelled == len(dev), "Some dev rows are not labelled. Finish dev labelling first."
assert eval_labelled == 0, "Eval rows have labels. Stop: the eval rows must stay blank for this step."
print("Sheet check passed.")

columns: ['pair_id', 'role', 'post_id', 'post_text', 'rank', 'score', 'passage', 'source', 'url', 'relevant_0_or_1']
role
dev     120
eval    120
dev rows labelled: 120 of 120
eval rows labelled: 0 (must be 0)
Sheet check passed.


## Cell 5: run scripts/14 (output shown)

The script runs the BM25 and the hybrid retriever on each dev post, and writes the overlap summary and the list of unlabelled passages.

In [5]:
import os, subprocess, sys
r = subprocess.run([sys.executable, "scripts/14_dev_retriever_overlap.py"],
                   cwd=PROJECT, capture_output=True, text=True)
print(r.stdout[-4000:])
if r.returncode != 0:
    print("---- stderr ----")
    print(r.stderr[-4000:])
r.check_returncode()

{
  "posts": 40,
  "hybrid_pairs": 120,
  "hybrid_pairs_already_labelled": 49,
  "unlabelled_hybrid_pairs": 71,
  "posts_with_new_pair": 39,
  "alpha": 0.5,
  "top_k": 3,
  "embed_model": "intfloat/multilingual-e5-base",
  "device": "cuda",
  "note": "Pair-level: a (post, passage) pair counts as labelled only if that post was labelled with that passage. Passages are matched by their first 400 characters. Dev labels only."
}
New (post, passage) pairs to label: 71 (see /content/drive/MyDrive/BFND-data/results/evidence/dev_new_passages.csv)



## Cell 6: the result

In [6]:
import json
import pandas as pd

RESULTS = f"{BFND}/results/evidence"
summary = json.load(open(f"{RESULTS}/dev_overlap.json", encoding="utf-8"))
print(json.dumps(summary, ensure_ascii=False, indent=2))

new_path = f"{RESULTS}/dev_new_passages.csv"
new = pd.read_csv(new_path, encoding="utf-8-sig", dtype=str, keep_default_na=False)
print(f"\nUnlabelled passages from the hybrid retriever: {len(new)}")
if len(new) == 0:
    print("Every hybrid passage is already labelled. No new dev labels are needed for the comparison.")
else:
    print("These need dev labels before the retrievers can be compared (D-11).")
    print(new[["post_id", "source", "passage"]].head(10).to_string(index=False)[:3000])

{
  "posts": 40,
  "hybrid_pairs": 120,
  "hybrid_pairs_already_labelled": 49,
  "unlabelled_hybrid_pairs": 71,
  "posts_with_new_pair": 39,
  "alpha": 0.5,
  "top_k": 3,
  "embed_model": "intfloat/multilingual-e5-base",
  "device": "cuda",
  "note": "Pair-level: a (post, passage) pair counts as labelled only if that post was labelled with that passage. Passages are matched by their first 400 characters. Dev labels only."
}

Unlabelled passages from the hybrid retriever: 71
These need dev labels before the retrievers can be compared (D-11).
                             post_id           source                                                                                                                                                                                                                                                                                                                                                                                                          passag

## Cell 7: copy the results to Drive (copy only, nothing deleted)

In [7]:
import time, shutil
stamp = time.strftime("%Y%m%d_%H%M")
EXPORT = f"{BFND}/exports/step5_{stamp}"
os.makedirs(EXPORT, exist_ok=True)
shutil.copytree(f"{BFND}/results/evidence", f"{EXPORT}/evidence", dirs_exist_ok=True)
print("Exported to:", f"{EXPORT}/evidence")

Exported to: /content/drive/MyDrive/BFND-data/exports/step5_20261126_1940/evidence
